In [2]:
import pandas as pd

corpus_mansi = pd.read_csv('synthetic_dataset/corpus_mansi.csv')
corpus_mansi

,text
0,мань кол
1,"И ти пилта̄л, ёмас э̄лмхо̄ласыт, тиныӈ совуп ӯ..."
2,Сысэ̄т о̄щнэ пāйпе аквматэ̄рт тāрви­тыӈыг емту...
3,Ко̄лпал хурит ко̄л ко̄лкан котильт а̄рась э̄тэ...
4,"Вāим, хотал сыс тэ̄тал ты ёмантас: «Я-та, то̄в..."
...,...
176136,Сун сысэ̄н нэ̄гим ёралахтым ёралахтым ёл-хуяс
176137,Са̄лыянын вуен!
176138,"Яныг, сака яныг ё̄рн ма̄хум кол о̄лыс"
176139,"Сун сихыртме мус, хунь ё̄р Вырпаду ва̄ӈнэт то̄..."


In [3]:
dataset = pd.DataFrame(columns=['audio_file', 'text'])

for i in range(0, 76132):
    sample = corpus_mansi.iloc[i]

    new_row = pd.DataFrame([{'audio_file': f'{i}.wav', 'text': sample['text']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)


dataset

,audio_file,text
0,0.wav,мань кол
1,1.wav,"И ти пилта̄л, ёмас э̄лмхо̄ласыт, тиныӈ совуп ӯ..."
2,2.wav,Сысэ̄т о̄щнэ пāйпе аквматэ̄рт тāрви­тыӈыг емту...
3,3.wav,Ко̄лпал хурит ко̄л ко̄лкан котильт а̄рась э̄тэ...
4,4.wav,"Вāим, хотал сыс тэ̄тал ты ёмантас: «Я-та, то̄в..."
...,...,...
76127,76127.wav,Сыс ё̄р тувл кинсэ̄в.
76128,76128.wav,Нас вос суссылтанувлын.
76129,76129.wav,Нэ̄мхоты ла̄глыӈ на̄йын ат ва̄йвес.
76130,76130.wav,Сэ̄мыл йивпа ялпыӈ тумпын котьлэ̄т


In [4]:
dataset.to_csv('synthetic_dataset/metadata.csv', index=False, sep="\t")

In [5]:
import pandas as pd
import regex as re
import os
import unicodedata

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv('synthetic_dataset/metadata.csv', delimiter="\t")
# df

In [6]:
folder_path = 'synthetic_dataset/wavs'
file_count = sum(1 for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)))
print(f"Количество файлов в папке 'wavs': {file_count}")
print(f"Количество записей в df: {len(df)}")

Количество файлов в папке 'wavs': 76132
Количество записей в df: 76132


In [7]:
wavs_list = os.listdir(folder_path)

to_drop = set()
for idx, row in df.iterrows():
    if row['audio_file'] not in wavs_list:
        to_drop.add(idx)

lat_to_cyr = {
    "a": "а", "A": "А",
    "e": "е", "E": "Е",
    "i": "и", "I": "И",
    "o": "о", "O": "О",
    "u": "у", "U": "У",
    "y": "ы", "Y": "Ы",
}

punct_map = {
        '“': '"', '”': '"', '„': '"', '«': '"', '»': '"',
        '‘': "'", '’': "'", '‚': "'",
        '—': '-', '–': '-', '−': '-',
    }

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFD", str(text))  # 1. разложение
    
    for lat, cyr in lat_to_cyr.items():
        text = text.replace(lat, cyr)
    
    text = unicodedata.normalize("NFC", text)

    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')

    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    for k, v in punct_map.items():
        text = text.replace(k, v)
    
    return text
    
df['text'] = df['text'].apply(normalize_text)

In [8]:
df = df.drop(to_drop)

df = df.drop_duplicates(subset=['text'])
df = df.dropna()

print(f"Количество записей в df: {len(df)}")

Количество записей в df: 73629


In [9]:
wav_to_drop = []

audio_list = df['audio_file'].tolist()
for wav in wavs_list:
    if wav not in audio_list:
        wav_to_drop.append(wav)

print(f"Количество файлов в папке 'wavs': {len(wavs_list) - len(wav_to_drop)}")

Количество файлов в папке 'wavs': 73629


In [10]:
folder_path = 'synthetic_dataset/wavs'

for wav in wav_to_drop:
    file_path = os.path.join(folder_path, wav)
    if os.path.isfile(file_path):
        os.remove(file_path)

print(f"{len(wav_to_drop)} файлов удалено")

df.to_csv('synthetic_dataset/metadata.csv', index=False, sep="\t")

2503 файлов удалено
